# Cell Image Labeling Notebook

This notebook processes fluorescence microscopy images to generate YOLO-format labels for cell detection.
It crops images with overlapping regions and creates corresponding bounding box annotations.

## Workflow
1. Select datasets and variables
2. Initialize output directories
3. Process each image with sliding window crops
4. Generate YOLO format labels and save annotations

## Imports

In [35]:
import cv2
import pandas as pd
import numpy as np
import os
from CellProcessor import read_image, process_image, get_bboxes, get_label_yolo, list_dataset, use_dataset, list_variables, use_variables

## Configuration
Select the dataset and preprocessing parameters to use.

In [36]:
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


In [15]:
print("Available preprocessing variable sets:")
list_variables()

Available preprocessing variable sets:


,ID,Time,Brightness,Contrast,Threshold,Erosion,Dialation,Description
0,1,2023-12-20T14:28:27,0.000000,1.700000,32,1,1,NaN
1,2,2023-12-20T14:31:46,-8.151099,4.527473,38,1,3,NaN
2,3,2024-01-05T09:56:59,0.000000,5.252747,28,1,3,NaN
3,4,2024-01-05T10:18:01,-13.049451,9.208791,40,3,5,NaN
4,5,2024-01-08T09:11:40,-9.359890,5.565934,45,3,1,NaN
5,6,2024-01-08T10:02:00,-9.453297,5.401099,45,2,3,NaN
6,7,2024-01-08T10:40:07,-9.359890,5.598901,42,2,1,NaN
7,8,2024-01-08T11:08:38,-9.406593,5.549451,42,2,3,NaN
8,9,2024-01-31T11:37:07,-7.304945,8.483516,42,2,3,NaN
9,10,2024-02-01T13:11:22,-9.400000,7.379121,42,2,3,NaN


## Setup Output Directories

Load the dataset and create output directories for labeled images, masks, and cropped images.

In [51]:
# Load dataset configuration
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}")

# Construct input paths
GREEN_PATH = os.path.join(dataset['Image_path'], dataset['Death_type'], f"{dataset['Cell_type']}_Green/")
PHASE_PATH = os.path.join(dataset['Image_path'], dataset['Death_type'], f"{dataset['Cell_type']}_Phase/")

# Construct output paths
base_output = os.path.join(dataset['Image_path'], dataset['Death_type'])
LABELED_IMAGES_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Labeles_phase/")
IMAGES_MASKS_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Masks_phase/")
IMAGES_PHASE_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Phase_Crop/")
IMAGES_GREEN_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Green_Crop/")

# Create output directories
output_dirs = [
    (LABELED_IMAGES_DIR_PATH, "Labels"),
    (IMAGES_MASKS_DIR_PATH, "Masks"),
    (IMAGES_PHASE_PATH, "Phase crops"),
    (IMAGES_GREEN_PATH, "Green crops")
]

for dir_path, dir_name in output_dirs:
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"✓ {dir_name} directory ready: {dir_path}")
    except OSError as e:
        print(f"✗ Error creating {dir_name} directory: {e}")

Dataset: MEF cells, Necroptosis death type
Image path: Data
✓ Labels directory ready: Data/Necroptosis/MEF_Labeles_phase/
✓ Masks directory ready: Data/Necroptosis/MEF_Masks_phase/
✓ Phase crops directory ready: Data/Necroptosis/MEF_Phase_Crop/
✓ Green crops directory ready: Data/Necroptosis/MEF_Green_Crop/


## Process Images and Generate Labels

For each input image:
1. Create overlapping crops (sliding window)
2. Apply preprocessing filters
3. Detect cell bounding boxes
4. Generate YOLO-format annotations
5. Save crops and labels

### Crop Parameters
- **Crop size**: 128×128 pixels
- **Overlap**: 64×52 pixels (horizontal×vertical)
- **Step size**: 64×76 pixels

In [31]:
variables = use_variables(1)
variables

{'ID': 1,
 'Time': '2023-12-20T14:28:27',
 'Brightness': -18.2,
 'Contrast': 12.0,
 'Threshold': 42,
 'Erosion': 2,
 'Dialation': 3,
 'Description': nan}

In [ ]:
# Polished processing loop: clearer names, safety checks, and robust label filtering
CROP_WIDTH = 128
CROP_HEIGHT = 128
CROP_WIDTH_OVERLAP = 64
CROP_HEIGHT_OVERLAP = 52
CROP_WIDTH_SHIFT = CROP_WIDTH - CROP_WIDTH_OVERLAP
CROP_HEIGHT_SHIFT = CROP_HEIGHT - CROP_HEIGHT_OVERLAP

# thresholds
MAX_SIGNAL_THRESHOLD = 40.0   # percent
MIN_DETECTION_SIZE = 0.01     # minimal box side pixels
images = sorted([f for f in os.listdir(GREEN_PATH) if f.lower().endswith(('png','jpg','jpeg'))])
total_images = len(images)

if total_images == 0:
    print(f"⚠ No images found in {GREEN_PATH}")

processed_count = 0
label_count = 0
error_count = 0

for idx, image in enumerate(images, start=1):
    try:
        print(f"[{idx}/{total_images}] {image}", end=" ... ")
        green_image_path = os.path.join(GREEN_PATH, image)
        phase_image_path = os.path.join(PHASE_PATH, image)

        img, img_base = read_image(green_image_path, phase_image_path)
        img_green = cv2.imread(green_image_path, cv2.IMREAD_UNCHANGED)

        if img is None or img_base is None:
            print("✗ Failed to load image(s)")
            error_count += 1
            continue

        img_base_h, img_base_w = img_base.shape[:2]

        num_width_windows = max(1, (img_base_w - CROP_WIDTH) // CROP_WIDTH_SHIFT)
        num_height_windows = max(1, (img_base_h - CROP_HEIGHT) // CROP_HEIGHT_SHIFT)

        crops_generated = 0

        for w_idx in range(int(num_width_windows)):
            for h_idx in range(int(num_height_windows)):
                x0 = int(w_idx * CROP_WIDTH_SHIFT)
                y0 = int(h_idx * CROP_HEIGHT_SHIFT)

                img_crop = img[y0:y0 + CROP_HEIGHT, x0:x0 + CROP_WIDTH]
                img_base_crop = img_base[y0:y0 + CROP_HEIGHT, x0:x0 + CROP_WIDTH]
                img_green_crop = img_green[y0:y0 + CROP_HEIGHT, x0:x0 + CROP_WIDTH]

                # ensure full-size crop
                if img_crop.shape[0] < CROP_HEIGHT or img_crop.shape[1] < CROP_WIDTH:
                    continue

                img_processed = process_image(
                    img_crop,
                    variables['Contrast'],
                    variables['Brightness'],
                    variables['Threshold'],
                    variables['Erosion'],
                    variables['Dialation'],
                    plot=False
                )

                bboxes = get_bboxes(img_processed)
                if not bboxes:
                    continue

                # compute mean signal as percentage (robust to image bit-depth)
                mean_signal = (img_processed.sum() / (img_processed.size * 255.0)) * 100.0
                if mean_signal >= MAX_SIGNAL_THRESHOLD:
                    continue

                label_boxes = []
                crop_h, crop_w = img_processed.shape[:2]

                for start_pt, end_pt in bboxes:
                    # skip invalid coordinates
                    if start_pt[0] < 0 or start_pt[1] < 0 or end_pt[0] < 0 or end_pt[1] < 0:
                        continue

                    nx, ny, nw, nh = get_label_yolo(start_pt, end_pt, crop_w, crop_h)

                    # keep only reasonably sized detections
                    if nw >= MIN_DETECTION_SIZE and nh >= MIN_DETECTION_SIZE:
                        label_boxes.append(f"0 {nx} {ny} {nw} {nh}\n")

                if label_boxes:
                    base_name = os.path.splitext(image)[0]
                    suffix = f"{w_idx}_{h_idx}"

                    img_mask_file = os.path.join(IMAGES_MASKS_DIR_PATH, f"{base_name}_{suffix}.png")
                    img_file = os.path.join(IMAGES_PHASE_PATH, f"{base_name}_{suffix}.png")
                    img_green_file = os.path.join(IMAGES_GREEN_PATH, f"{base_name}_{suffix}.png")
                    label_file = os.path.join(LABELED_IMAGES_DIR_PATH, f"{base_name}_{suffix}.txt")

                    cv2.imwrite(img_mask_file, img_processed)
                    cv2.imwrite(img_file, img_base_crop)
                    cv2.imwrite(img_green_file, img_green_crop)

                    with open(label_file, "w") as ann:
                        ann.writelines(label_boxes)

                    crops_generated += 1
                    label_count += len(label_boxes)

        processed_count += 1
        print(f"✓ ({crops_generated} crops)")

    except Exception as e:
        print(f"✗ Error: {e}")
        error_count += 1

print("\n" + "="*50)
print("Processing complete:")
print(f"  Images processed: {processed_count}/{total_images}")
print(f"  Errors: {error_count}")
print(f"  Total labels generated: {label_count}")
print("="*50)

[1/13] VID856_B4_1_00d00h00m.png ... ✓ (78 crops)
[2/13] VID856_B4_1_00d02h00m.png ... ✓ (167 crops)
[3/13] VID856_B4_1_00d04h00m.png ... ✓ (218 crops)
[4/13] VID856_B4_1_00d06h00m.png ... ✓ (239 crops)
[5/13] VID856_B4_1_00d08h00m.png ... ✓ (240 crops)
[6/13] VID856_B4_1_00d10h00m.png ... ✓ (240 crops)
[7/13] VID856_B4_1_00d12h00m.png ... ✓ (240 crops)
[8/13] VID856_B4_1_00d14h00m.png ... ✓ (236 crops)
[9/13] VID856_B4_1_00d16h00m.png ... ✓ (238 crops)
[10/13] VID856_B4_1_00d18h00m.png ... ✓ (237 crops)
[11/13] VID856_B4_1_00d20h00m.png ... ✓ (226 crops)
[12/13] VID856_B4_1_00d22h00m.png ... ✓ (213 crops)
[13/13] VID856_B4_1_01d00h00m.png ... ✓ (219 crops)

Processing complete:
  Images processed: 13/13
  Errors: 0
  Total labels generated: 14282
